In [1]:
from pathlib import Path
from mutools.plotting import run, prism_schematic, use_style
import mutools.plotting as mp

# Configure once — applies everywhere
mp.saver.configure(fmt="pdf", dpi=300, rasterized=True, ytick_precision=2)

use_style("rootlike")
INPUT = Path('/Users/mueller/Projects/spineosc')
#OUTPUT = Path('/Users/mueller/Projects/spineosc_workshop2026/figures/v1.0.9_rev0')
TECHNOTE = Path('/Users/mueller/Projects/spineosc_technote/figures/systematics')

def map_file(
    figure_directory: Path,
    out: Path,
    map: dict[str, str]
) -> None:
    """
    This function takes a `figure_directory`, an `out` directory, and a
    `map` of old filenames to new filenames. It renames the files in
    the `figure_directory` according to the `map` and saves them in the
    `out` directory. If a file in the `map` does not exist in the
    `figure_directory`, it prints a warning message.

    Parameters
    ----------
    figure_directory : Path
        The directory where the original figure files are located.
    out : Path
        The directory where the renamed figure files should be saved.
    map : dict[str, str]
        A dictionary mapping old filenames (keys) to new filenames
        (values). The old filenames should be relative to the
        `figure_directory`, and the new filenames should be relative to
        the `out` directory.

    Returns
    -------
    None.
    """
    for old, new in map.items():
        # We construct the full paths to the old and new files. The
        # `old_path` is the `figure_directory` + the `old` filename,
        # and the `new_path` is the `out` directory + the `new`
        # filename.
        old_path = out / old
        new_path = figure_directory / new

        # The `new_path` is some directory + filename, so we need to
        # make sure the directory exists before we can rename the file.
        new_path.parent.mkdir(parents=True, exist_ok=True)

        # Now we can rename the file if it exists. If it doesn't exist,
        # we print a warning message.
        if old_path.exists():
            old_path.rename(new_path)
            print(f"Renamed {old_path} to {new_path}")
        else:
            print(f"Warning: {old_path} does not exist and cannot be renamed to {new_path}")

In [2]:
# Systematic-uncertainty breakdowns for the technote, per detector and channel,
# in reconstructed and true neutrino energy.
#
# Inputs are PROfit `plot` outputs of xml/contours/contours_base.xml run with
# --fit-variable 3 (Reco E) and --fit-variable 2 (True E); PROfit only writes
# the fractional-systematic tree for the fitting variable.
#
# Broad breakdown (one line per type, quadrature total):
#   Flux                      Flux + G4 (beamline Geant4 reinteractions)
#   Cross Section             every Cross-Section-* category
#   Uncorrelated Systematics  Detector Systematics + POT Normalization + CORSIKA
#   Prediction Stat.          MC statistics
# Narrow breakdowns show each type's components with their quadrature total.
# Prediction stat. is a single component, so it has no narrow breakdown.
#
# Totals are drawn from the plotted components (show_total), NOT PROfit's
# TOTAL entry: the mcstat allowlist carries two tags ("stat,MC Stats") and
# PROfit's TOTAL counts that one covariance twice.

VARIABLES = {
    # token: (histogram file, x label, [single-proton xlim, multi-proton xlim])
    'recoe': (INPUT / 'systs_recoE_PROplot_histograms.root', 'Reconstructed Neutrino Energy [GeV]', [[0.3, 1.5], [0.4, 2.0]]),
    'truee': (INPUT / 'systs_trueE_PROplot_histograms.root', 'True Neutrino Energy [GeV]',          [[0.3, 3.0], [0.3, 3.0]]),
}

# (legend label, file token, suffix of this detector's systematic names)
DETECTORS = [
    ('SBND Simulation',         'sbnd_run1',   'sbnd_Run1'),
    ('ICARUS Run 2 Simulation', 'icarus_run2', 'icarus_Run2'),
    ('ICARUS Run 4 Simulation', 'icarus_run4', 'icarus_Run4'),
]
CHANNELS = [('Single-Proton', 'singlep'), ('Multi-Proton', 'multip')]

XSEC = {
    'Cross-Section-QE':           'QE',
    'Cross-Section-MEC':          'MEC',
    'Cross-Section-RES':          'RES',
    'Cross-Section-DIS':          'DIS',
    'Cross-Section-FSI':          'FSI',
    'Cross-Section-COH':          'COH',
    'Cross-Section-NCEL':         'NC Elastic',
    'Cross-Section-NonRES-nu':    r'Non-RES $\nu$',
    'Cross-Section-NonRES-nubar': r'Non-RES $\bar{\nu}$',
}

FLUX = {
    'Flux:piplus_Flux':            r'$\pi^{+}$ Production',
    'Flux:piminus_Flux':           r'$\pi^{-}$ Production',
    'Flux:kplus_Flux':             r'$K^{+}$ Production',
    'Flux:kminus_Flux':            r'$K^{-}$ Production',
    'Flux:kzero_Flux':             r'$K^{0}$ Production',
    'Flux:nucleoninexsec_Flux':    'Nucleon Inelastic XS',
    'Flux:nucleonqexsec_Flux':     'Nucleon QE XS',
    'Flux:nucleontotxsec_Flux':    'Nucleon Total XS',
    'Flux:pioninexsec_Flux':       'Pion Inelastic XS',
    'Flux:pionqexsec_Flux':        'Pion QE XS',
    'Flux:piontotxsec_Flux':       'Pion Total XS',
    'Flux:horncurrent_Flux':       'Horn Current',
    'Flux:expskin_Flux':           'Skin Depth',
}

GEANT = {
    'G4:reinteractions_proton_Geant4':  'G4 Proton Reinteractions',
    'G4:reinteractions_piplus_Geant4':  r'G4 $\pi^{+}$ Reinteractions',
    'G4:reinteractions_piminus_Geant4': r'G4 $\pi^{-}$ Reinteractions',
}


def uncorrelated_components(suffix):
    """This detector's detector-model, POT and (SBND only) CORSIKA components."""
    det = 'Detector Systematics'
    components = {}
    if suffix.startswith('sbnd'):
        components[f'{det}:ChargeScale15_{suffix}'] = 'Charge Scale / Resolution'
    else:
        components[f'{det}:ChargeScale_{suffix}'] = 'Charge Scale / Resolution'
        components[f'{det}:SCE_{suffix}'] = 'SCE'
    components[f'{det}:TriggerEmulation_{suffix}'] = 'Trigger Emulation'
    components[f'{det}:WireModxThetaXW_{suffix}'] = r'WireMod X-$\theta_{xw}$'
    components[f'{det}:WireModYZ_{suffix}'] = 'WireMod Y-Z'
    if suffix.startswith('sbnd'):
        components[f'{det}:DENT_{suffix}'] = 'DENT'
    components[f'POT Normalization:ToroidCalibration_{suffix}'] = 'Toroid Calibration'
    components[f'POT Normalization:ProtonMiss_{suffix}'] = 'Protons Missing Target'
    if suffix.startswith('sbnd'):
        components['Cosmic Background Normalization'] = 'CORSIKA Normalization'
    return components


def breakdowns(suffix):
    """name -> (tags, groups, label_map) for one detector."""
    broad = {
        'Flux': ['Flux', 'G4'],
        'Cross Section': list(XSEC),
        'Uncorrelated Systematics': ['Detector Systematics', 'POT Normalization', 'Cosmic Background Normalization'],
    }
    unc = uncorrelated_components(suffix)
    return {
        'summary':      (list(broad) + ['stat'], broad, {'stat': 'Prediction Stat.'}),
        'flux':         (list(FLUX), None, dict(FLUX)),
        'xsec':         (list(XSEC), None, dict(XSEC)),
        'geant':        (list(GEANT), None, dict(GEANT)),
        'uncorrelated': (list(unc), None, dict(unc)),
    }


In [3]:
# ---------------------------------------------------------------------------
# Y-AXIS RANGES AND LEGEND PLACEMENT -- set these by hand.
#
# One entry per breakdown category, applied to EVERY detector and to BOTH
# energy variables, so that plots which appear side by side in the technote
# share an axis. Run the report cell below to see the values in the files.
# ---------------------------------------------------------------------------

YLIM = {
    # category:        [ymin, ymax]        largest value in the files
    'summary':         [0.0, 0.80],        # 0.51
    'flux':            [0.0, 0.50],        # 0.30
    'geant':           [0.0, 0.25],        # 0.16
    'xsec':            [0.0, 0.65],        # 0.42
    'uncorrelated':    [0.0, 0.14],        # 0.089
}

# Optional exceptions, when one case genuinely should not share the range
# above. Keys are (category,), (category, variable) or
# (category, variable, channel), e.g.
#   ('uncorrelated', 'truee'): [0.0, 0.06],
#   ('summary', 'recoe', 'singlep'): [0.0, 0.5],
# The most specific matching key wins.
YLIM_OVERRIDES = {}

# Legend corner and column count per category.
LEGEND = {
    'summary':      ('upper left', 1),
    'flux':         ('upper left', 2),
    'geant':        ('upper left', 1),
    'xsec':         ('upper left', 2),
    'uncorrelated': ('upper left', 2),
}
DEFAULT_LEGEND = ('upper left', 2)  # used by any category not listed above


def ylim_for(category, variable, channel):
    """Most specific YLIM_OVERRIDES entry, else the category's YLIM.

    A category with no entry falls back to matplotlib's autoscale, which makes
    its panels disagree with each other -- the warning says which to add."""
    for key in ((category, variable, channel), (category, variable), (category,)):
        if key in YLIM_OVERRIDES:
            return YLIM_OVERRIDES[key]
    if category not in YLIM:
        print(f"WARNING: no YLIM entry for {category!r}; autoscaling (ranges will not match across plots)")
        return None
    return YLIM[category]


In [4]:
# Largest value plotted in each category (components and their total), over
# the x-range actually shown. Use it to choose the YLIM entries above: pick a
# ymax comfortably above the "all" column so the legend has room.
import numpy as np
from mutools.plotting.profit import ProfitPlotData, TraceType


def category_maxima():
    rows = {}
    for var_token, (input_file, _xlabel, xlims) in VARIABLES.items():
        data = ProfitPlotData(input_file)
        for detector, (_label, _token, suffix) in enumerate(DETECTORS):
            for channel, (_ch_label, ch_token) in enumerate(CHANNELS):
                lo, hi = xlims[channel]
                for name, (tags, groups, _lbl) in breakdowns(suffix).items():
                    values = []
                    for tag in tags:
                        members = groups[tag] if groups and tag in groups else [tag]
                        sq = None
                        for member in members:
                            t, _, sysname = member.partition(':')
                            a = data.get_trace(f"0:{detector}:{channel}:{detector}:{t}:{sysname or 'SUM'}", TraceType.FRAC_SYST)
                            inx = (a[:, 2] > lo) & (a[:, 1] < hi)
                            v = a[inx, 3]
                            sq = v ** 2 if sq is None else sq + v ** 2
                        values.append(np.sqrt(sq))
                    total = np.sqrt(sum(v ** 2 for v in values))
                    peak = max(total.max(), max(v.max() for v in values))
                    rows.setdefault(name, {}).setdefault((var_token, ch_token), []).append(peak)
    for name, per_case in rows.items():
        overall = max(max(v) for v in per_case.values())
        print(f"{name:14s} all: {overall:.3f}   ylim in use: {YLIM.get(name, 'not set -- autoscaled')}")
        for (var_token, ch_token), peaks in per_case.items():
            print(f"    {var_token:6s} {ch_token:8s} per detector: {[round(p, 3) for p in peaks]}")


category_maxima()


summary        all: 0.510   ylim in use: [0.0, 0.8]
    recoe  singlep  per detector: [np.float64(0.274), np.float64(0.222), np.float64(0.217)]
    recoe  multip   per detector: [np.float64(0.404), np.float64(0.379), np.float64(0.383)]
    truee  singlep  per detector: [np.float64(0.429), np.float64(0.398), np.float64(0.343)]
    truee  multip   per detector: [np.float64(0.47), np.float64(0.51), np.float64(0.429)]
flux           all: 0.304   ylim in use: [0.0, 0.5]
    recoe  singlep  per detector: [np.float64(0.122), np.float64(0.132), np.float64(0.128)]
    recoe  multip   per detector: [np.float64(0.168), np.float64(0.19), np.float64(0.206)]
    truee  singlep  per detector: [np.float64(0.253), np.float64(0.256), np.float64(0.248)]
    truee  multip   per detector: [np.float64(0.304), np.float64(0.261), np.float64(0.301)]
xsec           all: 0.415   ylim in use: [0.0, 0.65]
    recoe  singlep  per detector: [np.float64(0.221), np.float64(0.187), np.float64(0.188)]
    recoe  multip 

In [5]:
# Render every breakdown: 3 detectors x 2 channels x 2 energy variables x
# 4 categories. Axis ranges and legend placement come from the cell above.
import shutil
import tempfile
import matplotlib.pyplot as plt
from matplotlib import colormaps

mp.saver.configure(mark_hadj=0.005)
PLOT_DIR = TECHNOTE
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def make_config(input_file, output, detector, channel, xlabel, xlim, ylim,
                tags, groups, label_map, legend_loc, legend_ncol):
    """Configuration for ONE uncertainty plot."""
    # Explicit qualitative palettes: the style's default cycle includes faint
    # greys that are hard to tell apart on a many-line plot.
    cmap = colormaps['tab10' if len(tags) <= 10 else 'tab20']
    plot = {
        'type': 'uncertainty',
        'detectors': [detector],
        'channel': channel,
        'tags': tags,
        'show_total': True,
        'xlabel': xlabel,
        'ylabel': 'Fractional Uncertainty',
        'xlim': xlim,
        'ylim': ylim,
        'watermark': r'$\bf{SBN}$ Internal',
        'legend_title': f'{DETECTORS[detector][0]}\n{CHANNELS[channel][0]}',
        'label_map': {**label_map, 'Total': 'Total Uncertainty'},
        'legend_loc': legend_loc,
        'legend_ncol': legend_ncol,
        'colors': [cmap(i) for i in range(len(tags))],
    }
    if groups:
        plot['groups'] = groups
    return {
        'general': {
            'input': str(input_file),
            'output': str(output),
            'savefig': True,
            'code_version': 'v3.0.0',
            'selection_version': 'v1.0.10',
            'subchannels': ['Target $\\nu_\\mu$ CC', 'Other $\\nu_\\mu$ CC', '$\\nu_e$ CC', '$\\nu$ NC', 'Out-of-time Cosmics', 'In-time Cosmics'],
            'detectors': [d[0] for d in DETECTORS],
            'channels': [c[0] for c in CHANNELS],
        },
        'plot': [plot],
    }


with tempfile.TemporaryDirectory() as staging:
    staging = Path(staging)
    for var_token, (input_file, xlabel, xlims) in VARIABLES.items():
        for detector, (det_label, det_token, suffix) in enumerate(DETECTORS):
            for channel, (ch_label, ch_token) in enumerate(CHANNELS):
                for name, (tags, groups, label_map) in breakdowns(suffix).items():
                    legend_loc, legend_ncol = LEGEND.get(name, DEFAULT_LEGEND)
                    run(make_config(input_file, staging, detector, channel, xlabel,
                                    xlims[channel], ylim_for(name, var_token, ch_token),
                                    tags, groups, label_map, legend_loc, legend_ncol))
                    plt.close('all')
                    target = PLOT_DIR / f'systematics_{det_token}_{name}_{var_token}_{ch_token}.pdf'
                    shutil.move(staging / f'uncertainty_{detector}_v3.0.0.pdf', target)
    n_categories = len(breakdowns(DETECTORS[0][2]))
    print(f'wrote {n_categories * len(VARIABLES) * len(DETECTORS) * len(CHANNELS)} figures to {PLOT_DIR}')


wrote 60 figures to /Users/mueller/Projects/spineosc_technote/figures/systematics
